In [3]:
!pip install python-telegram-bot==20.7 transformers torch pillow

# Fix Runtime Error: Cannot close a running event loop
import nest_asyncio
nest_asyncio.apply()

import logging
import torch
from PIL import Image
from io import BytesIO
from transformers import AutoImageProcessor, ViTForImageClassification
from telegram import Update
from telegram.ext import ApplicationBuilder, MessageHandler, CommandHandler, filters, ContextTypes

# -------------------------------------
# Load Model (Vision Transformer)
# -------------------------------------
model_name = "google/vit-base-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(model_name)

# -------------------------------------
# Logging (optional)
# -------------------------------------
logging.basicConfig(
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    level=logging.INFO
)

# -------------------------------------
# Start Command
# -------------------------------------
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("Hi! Send me an image and I will classify it using ViT.")

# -------------------------------------
# Image Handler
# -------------------------------------
async def handle_image(update: Update, context: ContextTypes.DEFAULT_TYPE):
    photo = update.message.photo[-1]  # highest resolution
    file = await photo.get_file()
    image_bytes = await file.download_as_bytearray()

    # Convert bytes → PIL image
    img = Image.open(BytesIO(image_bytes)).convert("RGB")

    # Preprocess
    inputs = image_processor(images=img, return_tensors="pt")

    # Run model
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    predicted_idx = logits.argmax(-1).item()
    predicted_label = model.config.id2label[predicted_idx]

    await update.message.reply_text(f"Prediction: **{predicted_label}**")

# -------------------------------------
# Main function
# -------------------------------------
from google.colab import userdata

async def main():
    BOT_TOKEN = userdata.get('TELEGRAM_API_KEY')

    app = ApplicationBuilder().token(BOT_TOKEN).build()

    app.add_handler(CommandHandler("start", start))
    app.add_handler(MessageHandler(filters.PHOTO, handle_image))

    print("🤖 Bot running… Send an image to your bot!")
    await app.run_polling()

# Entry point
if __name__ == "__main__":
    import asyncio
    asyncio.run(main())


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


🤖 Bot running… Send an image to your bot!


RuntimeError: Cannot close a running event loop